<a href="https://colab.research.google.com/github/Namesakenberg/deep_learning/blob/main/VGG16_implementation_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [4]:
!kaggle datasets download -d jtiptj/chest-xray-pneumoniacovid19tuberculosis

Dataset URL: https://www.kaggle.com/datasets/jtiptj/chest-xray-pneumoniacovid19tuberculosis
License(s): other
 99% 1.73G/1.74G [00:26<00:00, 124MB/s] 
100% 1.74G/1.74G [00:26<00:00, 70.8MB/s]


In [5]:
import zipfile
zip_ref = zipfile.ZipFile('/content/chest-xray-pneumoniacovid19tuberculosis.zip','r')
zip_ref.extractall('/content/chest-xray-data')
zip_ref.close()

In [6]:
train_dir = '/content/chest-xray-data/train'
test_dir = '/content/chest-xray-data/test'
val_dir = '/content/chest-xray-data/val'

In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen =  ImageDataGenerator(
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        fill_mode='nearest')
test_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    directory=train_dir,
    target_size=(224,224),
    color_mode='rgb',
    classes=None,
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
)
test_data = test_datagen.flow_from_directory(
    directory=test_dir,
    target_size=(224,224),
    color_mode='rgb',
    classes=None,
    class_mode='categorical',
    batch_size=32,
    shuffle=False,
)
val_data = val_datagen.flow_from_directory(
    directory=val_dir,
    target_size=(224,224),
    color_mode='rgb',
    classes=None,
    class_mode='categorical',
    batch_size=32,
    shuffle=False,
)


Found 6326 images belonging to 4 classes.
Found 771 images belonging to 4 classes.
Found 38 images belonging to 4 classes.


In [10]:
num_classes = train_data.num_classes
num_classes

4

In [12]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models,layers
from tensorflow.keras import Sequential

vgg_16 = keras.applications.VGG16(
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    input_shape=(224,224,3),
)
vgg_16.trainable = False
model = Sequential()
model.add(vgg_16)
model.add(layers.Flatten())
model.add(layers.Dense(128,activation ='relu'))
model.add(layers.Dense(64,activation='relu'))
model.add(layers.Dense(num_classes,activation='softmax'))

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

earlystopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    min_delta=0.001,
    patience=5,
    verbose=0,
    mode="auto",
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0,
)

model.fit(train_data,epochs=20,validation_data=val_data,callbacks=[earlystopping])

Epoch 1/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 144s 715ms/step - accuracy: 0.7769 - loss: 0.6426 - val_accuracy: 0.9211 - val_loss: 0.3061
Epoch 2/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 139s 698ms/step - accuracy: 0.8942 - loss: 0.2639 - val_accuracy: 0.7895 - val_loss: 0.3762
Epoch 3/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 137s 692ms/step - accuracy: 0.9157 - loss: 0.2237 - val_accuracy: 0.8947 - val_loss: 0.2882
Epoch 4/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 137s 690ms/step - accuracy: 0.9218 - loss: 0.2022 - val_accuracy: 0.8421 - val_loss: 0.5775
Epoch 5/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 138s 698ms/step - accuracy: 0.9206 - loss: 0.2037 - val_accuracy: 0.8947 - val_loss: 0.2987
Epoch 6/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 136s 683ms/step - accuracy: 0.9355 - loss: 0.1748 - val_accuracy: 0.8947 - val_loss: 0.1898
Epoch 7/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 136s 686ms/step - accuracy: 0.9344 - loss: 0.1698 - val_accuracy: 0.8684 - val_loss: 0.3391
Epoch 8/20
198/198 ━━━━━━━━━━━━━━━━━━━━ 136s 687ms/step - accuracy: 0.9387 -

In [14]:
test_loss, test_accuracy =  model.evaluate(test_data)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


25/25 ━━━━━━━━━━━━━━━━━━━━ 12s 414ms/step - accuracy: 0.9008 - loss: 0.2702
Test Accuracy: 91.57%
